# Bayesian state estimates of household WASH deprivation in Nigeria

This notebook fits a Bayesian area-level model to the survey-weighted state deprivation estimates and standard errors from the hotspot notebook. It estimates smoothed prevalence, state changes and uncertainty. The primary analysis fits **2018 and 2024**. A separate sensitivity fit includes **2013** using its complete-case definition.

Put this notebook beside **wash_hotspot_outputs**, which must contain **State_WASH_Deprivation_Prevalence.csv** and **State_Spatial_Neighbours.csv**. Then select **Run → Run All Cells** in JupyterLab. If packages are missing, run **%pip install numpy pandas scipy matplotlib pillow python-docx** and restart the kernel.

The maps use the same pinned Nigeria ADM1 boundary as the hotspot analysis. Put **geoBoundaries-NGA-ADM1.geojson** beside the notebook, or let the code download it. If the automatic download fails, [save this boundary file manually](https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1.geojson) with that filename.

Results go to **wash_bayesian_outputs**. LibreOffice Writer is required for automatic Word 97–2003 DOC export. If the converter is unavailable, open the saved DOCX in desktop Word and choose **Save As → Word 97–2003 Document (*.doc)**.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
from tempfile import TemporaryDirectory
import hashlib
import json
import os
import shutil
import subprocess

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import PathPatch
from matplotlib.path import Path as PlotPath
import numpy as np
import pandas as pd
from PIL import Image
from scipy.linalg import cho_factor, cho_solve
from scipy.special import expit, logit, logsumexp
from scipy.stats import halfnorm

ROOT = Path.cwd()
INPUT = ROOT / "wash_hotspot_outputs"
OUTPUT = ROOT / "wash_bayesian_outputs"
RATES_FILE = "State_WASH_Deprivation_Prevalence.csv"
LINKS_FILE = "State_Spatial_Neighbours.csv"
for filename in (RATES_FILE, LINKS_FILE):
    if not (INPUT / filename).is_file():
        raise FileNotFoundError(f"Missing {filename}. Run the hotspot notebook and keep its output folder beside this notebook.")

rates = pd.read_csv(INPUT / RATES_FILE)
required = {"state", "survey_round", "indicator", "weighted_pct", "se_pct", "weighted_outcome_known_pct", "review_flags"}
if missing := required - set(rates.columns):
    raise ValueError(f"Missing state estimate columns: {sorted(missing)}")
if rates.state.isna().any():
    raise ValueError("A row has no state name.")
states = sorted(rates.state.unique())
rounds = (2013, 2018, 2024)
keys = ["survey_round", "state"]
index = pd.MultiIndex.from_product([rounds, states], names=keys)
rates = rates.set_index(keys, verify_integrity=True)
if len(states) != 37 or "FCT" not in states or set(index) != set(rates.index):
    raise ValueError("Expected 37 state or FCT estimates for each of 2013, 2018 and 2024.")
rates = rates.reindex(index).reset_index()
expected = rates.survey_round.map({2013:"wash_dep_2plus_all_rounds", 2018:"wash_dep_2plus", 2024:"wash_dep_2plus"})
if not rates.indicator.eq(expected).all():
    raise ValueError("The input uses a different deprivation indicator from the hotspot notebook.")
p = rates.weighted_pct / 100
se = rates.se_pct / 100
if not (p.gt(0) & p.lt(1) & np.isfinite(se) & se.gt(0)).all():
    raise ValueError("Each input needs a proportion strictly between 0 and 1 and a positive finite survey SE.")
rates["review_flags"] = rates.review_flags.fillna("")
rates["direct_logit"] = logit(p)
rates["se_logit"] = se / (p * (1 - p))

links = pd.read_csv(INPUT / LINKS_FILE)
queen = links.loc[links.graph.eq("Queen"), ["state_1","state_2"]]
pairs = [tuple(sorted(pair)) for pair in queen.itertuples(index=False, name=None)]
if not pairs or len(pairs) != len(set(pairs)):
    raise ValueError("Missing or duplicated Queen links.")
state_id = {name:i for i,name in enumerate(states)}
n = len(states)
adj = np.zeros((n,n))
for a,b in pairs:
    if a == b or a not in state_id or b not in state_id:
        raise ValueError(f"Invalid Queen neighbour pair: {a}, {b}")
    i,j = state_id[a],state_id[b]
    adj[i,j] = adj[j,i] = 1
laplacian = np.diag(adj.sum(axis=1)) - adj
if np.count_nonzero(np.linalg.eigvalsh(laplacian) < 1e-8) != 1:
    raise ValueError("Queen graph must be connected.")
OUTPUT.mkdir(exist_ok=True)
print(f"Read {len(rates)} direct estimates and {len(pairs)} undirected Queen links.")


## 1. Fit the Bayesian model

The observed logit prevalence is modelled as normal around a latent state-round logit, with its design-based SE treated as fixed. The logit SE uses a first-order delta approximation. The latent logit is the sum of an intercept, a categorical survey-round effect, a spatial state effect shared across rounds, and an independent state-by-round deviation.

The spatial prior uses the inverse of the graph Laplacian plus 0.10 times the identity, then centres the spatial effect to sum to zero and scales its mean marginal variance to one. Intercept and round coefficients have zero-centred normal priors with SDs 2.5 and 1.0. The two random-effect SDs have independent half-normal priors with scale 1.0, represented by 21 grid points from 0.01 to 4.0 and probability masses from their surrounding bins. The model integrates over this finite grid and draws directly from the conditional Gaussian posterior. There are 8,000 independent mixture draws per fit.

The change for each state is 2024 minus 2018, calculated on the prevalence scale from paired posterior draws. The model treats survey sampling errors as independent between state-round estimates. It estimates deprivation among households with known WASH outcomes. Hatched areas retain the input review flags, including low coverage. Model uncertainty does not account for bias from missing outcomes. The categorical round model supports comparisons of observed rounds.


In [ ]:
from collections import defaultdict
from itertools import product

DRAWS = 8000
GRID = np.geomspace(.01, 4., 21)
cuts = np.r_[0., np.sqrt(GRID[:-1] * GRID[1:]), np.inf]
grid_mass = np.diff(halfnorm.cdf(cuts, scale=1.))
grid_pairs = list(product(range(len(GRID)), repeat=2))
rng = np.random.default_rng(20241110)

centering = np.eye(n) - np.ones((n,n)) / n
kernel = centering @ np.linalg.solve(laplacian + .10 * np.eye(n), centering)
kernel /= np.diag(kernel).mean()
eig, vectors = np.linalg.eigh(kernel)
root = vectors * np.sqrt(np.maximum(eig, 0))

def quantiles(draws):
    return np.quantile(draws, [.025, .50, .975], axis=0)

def fit_model(label, years):
    sample = rates.loc[rates.survey_round.isin(years)].sort_values(keys).reset_index(drop=True)
    y = sample.direct_logit.to_numpy()
    error = sample.se_logit.to_numpy()
    noise = error**2
    state_codes = sample.state.map(state_id).to_numpy()
    # 2018 is the reference year in both fits.
    extra_rounds = [year for year in years if year != 2018]
    X = np.column_stack([np.ones(len(y)), *[sample.survey_round.eq(yr).to_numpy() for yr in extra_rounds]])
    coef_sd = np.array([2.5, *[1.0 for _ in extra_rounds]])
    beta_cov = np.diag(coef_sd**2)
    cross_beta = beta_cov @ X.T
    cross_u = kernel[:, state_codes]
    spatial = cross_u[state_codes]
    base = X @ cross_beta

    def factor(i,j):
        cov = base + GRID[i]**2 * spatial + np.diag(noise + GRID[j]**2)
        return cho_factor(cov, lower=True, check_finite=False)

    log_weights = []
    for i,j in grid_pairs:
        cf = factor(i,j)
        log_likelihood = -.5 * (y @ cho_solve(cf,y,check_finite=False)
                                + 2 * np.log(np.diag(cf[0])).sum())
        log_weights.append(log_likelihood + np.log(grid_mass[i]) + np.log(grid_mass[j]))
    log_weights = np.array(log_weights)
    weights = np.exp(log_weights - logsumexp(log_weights))
    groups = defaultdict(list)
    for draw_id, pair_id in enumerate(rng.choice(len(grid_pairs), DRAWS, p=weights)):
        groups[pair_id].append(draw_id)

    latent = np.empty((DRAWS,len(y)))
    beta = np.empty((DRAWS,X.shape[1]))
    spatial_effect = np.empty((DRAWS,n))
    for pair_id, draw_ids in groups.items():
        i,j = grid_pairs[pair_id]
        s_spatial, s_interaction = GRID[i], GRID[j]
        count = len(draw_ids)
        # Draw prior components and correct them by conditioning on the observed direct logits.
        b = rng.normal(size=(count,X.shape[1])) * coef_sd
        u = (rng.normal(size=(count,n)) @ root.T) * s_spatial
        dev = rng.normal(scale=s_interaction,size=(count,len(y)))
        prior_theta = b @ X.T + u[:,state_codes] + dev
        simulated_y = prior_theta + rng.normal(size=prior_theta.shape) * error
        correction = cho_solve(factor(i,j), (y - simulated_y).T, check_finite=False).T
        beta[draw_ids] = b + correction @ cross_beta.T
        spatial_effect[draw_ids] = u + s_spatial**2 * correction @ cross_u.T
        latent[draw_ids] = prior_theta + correction @ base + s_spatial**2 * correction @ spatial + s_interaction**2 * correction

    prob = 100 * expit(latent)
    low, median, high = quantiles(prob)
    estimates = sample.drop(columns=["direct_logit","se_logit"]).assign(
        model=label, posterior_mean_pct=prob.mean(axis=0), posterior_median_pct=median,
        cri95_low_pct=low, cri95_high_pct=high, cri95_width_pp=high-low,
    )
    lookup = {(r.survey_round,r.state): i for i,r in enumerate(sample.itertuples(index=False))}
    change = (prob[:,[lookup[(2024,s)] for s in states]]
              - prob[:,[lookup[(2018,s)] for s in states]])
    low, median, high = quantiles(change)
    flagged = (sample.loc[sample.survey_round.isin([2018,2024])]
               .assign(flag=lambda x:x.review_flags.ne("")).groupby("state").flag.any())
    changes = pd.DataFrame({
        "model":label, "state":states, "mean_change_pp":change.mean(axis=0),
        "median_change_pp":median, "cri95_low_pp":low, "cri95_high_pp":high,
        "probability_decline":(change < 0).mean(axis=0),
        "input_review":flagged.reindex(states).to_numpy(),
    })

    predictive = latent + rng.normal(size=latent.shape) * error
    predictive_low, _, predictive_high = quantiles(predictive)
    diagnostics = {
        "model":label, "state_rounds":len(sample), "posterior_draws":DRAWS,
        "predictive_interval_coverage_pct":100*np.mean((y>=predictive_low)&(y<=predictive_high)),
        "mean_abs_direct_minus_smoothed_pp":np.abs(estimates.weighted_pct-estimates.posterior_mean_pct).mean(),
        "posterior_mass_grid_edge":weights[[i in (0,len(GRID)-1) or j in (0,len(GRID)-1) for i,j in grid_pairs]].sum(),
    }
    effects = []
    for effect,names,array in [("Spatial state effect", states, spatial_effect),
                               ("Round effect versus 2018", extra_rounds, beta[:,1:])]:
        lo,med,hi = quantiles(array)
        effects.extend({"model":label,"effect":effect,"name":name,"median_log_odds":m,
                        "cri95_low":l,"cri95_high":h}
                       for name,l,m,h in zip(names,lo,med,hi))
    hyper = pd.DataFrame([(label,GRID[i],GRID[j],w) for (i,j),w in zip(grid_pairs,weights)],
                         columns=["model","spatial_sd","state_round_sd","posterior_mass"])
    return estimates,changes,diagnostics,pd.DataFrame(effects),hyper

main, change, main_qc, main_effects, main_hyper = fit_model("Main 2018 and 2024", (2018,2024))
sensitivity, change_sensitivity, sens_qc, sens_effects, sens_hyper = fit_model("Includes 2013", (2013,2018,2024))
comparison = change[["state","mean_change_pp"]].merge(
    change_sensitivity[["state","mean_change_pp"]], on="state", validate="one_to_one",
    suffixes=("_main","_includes_2013"),
)
comparison["difference_pp"] = comparison.mean_change_pp_includes_2013 - comparison.mean_change_pp_main
diagnostics = pd.DataFrame([main_qc,sens_qc])
tables = {
    "Bayesian_State_Round_Prevalence.csv":pd.concat([main,sensitivity],ignore_index=True),
    "Bayesian_State_Change_2018_2024.csv":pd.concat([change,change_sensitivity],ignore_index=True),
    "Bayesian_State_Change_Sensitivity.csv":comparison,
    "Bayesian_Model_Effects.csv":pd.concat([main_effects,sens_effects],ignore_index=True),
    "Bayesian_Model_QC.csv":diagnostics,
    "Bayesian_Hyperparameter_Grid.csv":pd.concat([main_hyper,sens_hyper],ignore_index=True),
}
for name,table in tables.items():
    table.to_csv(OUTPUT/name,index=False)
display(diagnostics.round(3))


## 2. Map the fitted estimates and save the report

The first figure shows 2024 posterior mean deprivation and the width of its 95% credible interval. The second shows the 2018–2024 mean change and the posterior probability of a decline. Both use the main fit. Hatching marks direct estimates flagged for review in the previous notebook. The Word report includes a state table, checks and the two 600 dpi JPEG figures.

The predictive coverage check compares the fitted model's replications to the same state estimates used for fitting. Use it to diagnose model fit. Future predictive validation requires a separate exercise. The sensitivity CSV shows how including 2013 changes the posterior mean 2018–2024 differences.

Method references include [area-level estimation with survey variances](https://www150.statcan.gc.ca/n1/pub/12-001-x/2021002/article/00007-eng.htm) and [two-stage Bayesian estimation of proportions](https://doi.org/10.1111/insr.12572).


In [ ]:
BOUNDARY = ROOT/"geoBoundaries-NGA-ADM1.geojson"
URL = ("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/"
       "releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1.geojson")
CREDIT = "GRID3 Nigeria ADM1 via geoBoundaries gbOpen, 2022 boundaries, CC BY 4.0"
if not BOUNDARY.is_file():
    try:
        with urlopen(URL,timeout=45) as response:
            raw = response.read()
        json.loads(raw)
    except (OSError,ValueError) as error:
        raise RuntimeError(f"Save the boundary file from the manual link beside the notebook as {BOUNDARY.name}.") from error
    BOUNDARY.write_bytes(raw)
raw = BOUNDARY.read_bytes()
digest = hashlib.sha256(raw).hexdigest()
source_note = INPUT/"Boundary_Source.txt"
if source_note.is_file():
    hashes = [line.split()[-1] for line in source_note.read_text(encoding="utf-8").splitlines()
              if line.startswith("SHA-256 ")]
    if hashes and hashes != [digest]:
        raise ValueError("Map boundaries differ from those used in the hotspot run. Use the same full-resolution file.")
(OUTPUT/"Boundary_Source.txt").write_text(f"{CREDIT}\n{URL}\nSHA-256 {digest}\n",encoding="utf-8")
features = {}
for feature in json.loads(raw)["features"]:
    name = feature["properties"]["shapeName"]
    state = "FCT" if name == "Abuja Federal Capital Territory" else name
    if state in features:
        raise ValueError(f"Duplicate boundary for {state}.")
    features[state] = feature["geometry"]
if set(features) != set(states):
    raise ValueError("The boundary state names do not match the model results.")
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path



def polygon_path(geometry):
    kind = geometry["type"]
    if kind == "Polygon":
        polygons = [geometry["coordinates"]]
    elif kind == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        raise ValueError(f"Unexpected geometry type: {kind}")
    paths = []
    for polygon in polygons:
        for i, ring in enumerate(polygon):
            points = np.asarray(ring, dtype=float)[:, :2]
            if len(points) < 4 or not np.isfinite(points).all():
                raise ValueError("Malformed boundary ring.")
            if not np.array_equal(points[0], points[-1]):
                points = np.vstack([points, points[0]])
            x, y = points.T
            signed_area = np.sum(x[:-1] * y[1:] - y[:-1] * x[1:])
            # Opposite winding keeps interior holes open.
            if (signed_area > 0) != (i == 0):
                points = points[::-1]
            codes = np.full(len(points), PlotPath.LINETO, dtype=np.uint8)
            codes[0], codes[-1] = PlotPath.MOVETO, PlotPath.CLOSEPOLY
            paths.append(PlotPath(points, codes))
    return PlotPath.make_compound_path(*paths)

paths = {state: polygon_path(geometry) for state, geometry in features.items()}
coordinates = np.vstack([path.vertices for path in paths.values()])
west, south = coordinates.min(axis=0)
east, north = coordinates.max(axis=0)
if not (0 < west < east < 20 and 0 < south < north < 20):
    raise ValueError("Unexpected boundary extent. Use Nigeria ADM1 longitude and latitude.")
aspect = 1 / np.cos(np.deg2rad((south + north) / 2))


def draw(ax, frame, value, flag, cmap, norm, title, legend):
    for state, path in paths.items():
        row = frame.loc[state]
        ax.add_patch(PathPatch(path, facecolor=cmap(norm(row[value])), edgecolor="#41474E",
                               linewidth=.35,hatch="////" if row[flag] else None))
    ax.set(xlim=(west-.2,east+.2),ylim=(south-.18,north+.18),aspect=aspect)
    ax.set_axis_off()
    ax.set_title(title,fontsize=10)
    return plt.cm.ScalarMappable(norm=norm,cmap=cmap), legend

def map_pair(title, frame, specs, filename):
    fig, axes = plt.subplots(1,2,figsize=(7,3.65))
    fig.subplots_adjust(left=.035,right=.965,top=.83,bottom=.25,wspace=.10)
    fig.suptitle(title,y=.97,fontsize=13,fontweight="bold")
    for i,(ax,spec) in enumerate(zip(axes,specs)):
        scale,label = draw(ax,frame,*spec)
        bar = fig.colorbar(scale,cax=fig.add_axes([.095+i*.49,.20,.32,.026]),orientation="horizontal")
        bar.ax.tick_params(labelsize=7)
        bar.set_label(label,fontsize=8)
    fig.text(.5,.075,"Hatching indicates a flagged input estimate.",ha="center",fontsize=8)
    fig.text(.5,.028,CREDIT,ha="center",fontsize=6.5)
    return save_figure(fig,filename)

latest = main.loc[main.survey_round.eq(2024)].set_index("state").assign(flag=lambda x:x.review_flags.ne(""))
width = max(5,5*np.ceil(latest.cri95_width_pp.max()/5))
limit = max(5,5*np.ceil(change.mean_change_pp.abs().max()/5))
fig1 = map_pair("Modelled WASH deprivation in 2024",latest,[
    ("posterior_mean_pct","flag",plt.get_cmap("YlOrRd"),Normalize(0,100),"Posterior mean","Household percentage"),
    ("cri95_width_pp","flag",plt.get_cmap("Purples"),Normalize(0,width),"95% credible interval width","Percentage points"),
],"Map_Bayesian_Deprivation_2024_600dpi.jpg")
fig2 = map_pair("Modelled change in WASH deprivation",change.set_index("state"),[
    ("mean_change_pp","input_review",plt.get_cmap("RdBu_r"),TwoSlopeNorm(0,-limit,limit),
     "2024 minus 2018","Percentage points"),
    ("probability_decline","input_review",plt.get_cmap("Blues"),Normalize(0,1),
     "Probability of a decline","Posterior probability"),
],"Map_Bayesian_Deprivation_Change_600dpi.jpg")
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Bayesian state estimates of WASH deprivation in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")



doc.add_paragraph("Bayesian state estimates of WASH deprivation in Nigeria",style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey rounds 2018 and 2024")
doc.add_paragraph(
    "The model combines direct survey estimates and their standard errors across 36 states and the Federal "
    "Capital Territory. The main estimates use 2018 and 2024. The sensitivity analysis also includes the 2013 "
    "complete-case estimates."
)
doc.add_heading("Model checks",level=1)
qc_rows = [[r.model,r.state_rounds,f"{r.predictive_interval_coverage_pct:.1f}%",
            f"{r.mean_abs_direct_minus_smoothed_pp:.2f}",f"{100*r.posterior_mass_grid_edge:.2f}%"]
           for r in diagnostics.itertuples(index=False)]
add_table("Table 1. Posterior predictive and numerical checks",
          ["Fit","Area rounds","Predictive coverage","Mean absolute shift (pp)","Grid edge mass"],
          qc_rows,[1.55,.9,1.45,1.65,1.45],{0})
doc.add_paragraph(
    "Predictive coverage is an in-sample check. The graph prior smooths nearby states and the "
    "model allows state-specific deviations by survey round. Results apply to households with known "
    "WASH outcomes and treat the direct survey standard errors as fixed."
)
largest = comparison.difference_pp.abs().max()
doc.add_paragraph(f"Including 2013 changed the estimated 2018–2024 state differences by at most {largest:.2f} percentage points.")
doc.add_heading("State estimates and changes",level=1).paragraph_format.page_break_before = True
rows = []
for state in states:
    p = latest.loc[state]
    c = change.set_index("state").loc[state]
    flag = "†" if c.input_review else ""
    rows.append([state+flag,f"{p.posterior_mean_pct:.1f} ({p.cri95_low_pct:.1f}, {p.cri95_high_pct:.1f})",
                 f"{c.mean_change_pp:.1f} ({c.cri95_low_pp:.1f}, {c.cri95_high_pp:.1f})",
                 f"{100*c.probability_decline:.1f}%"])
add_table("Table 2. Main posterior means and 95% credible intervals",
          ["State or FCT","2024 deprivation (%)","Change since 2018 (pp)","Probability of decline"],
          rows,[1.55,1.75,2.00,1.70],{0},font_size=8.5)
doc.add_paragraph("† At least one main-round input was flagged for review.",style="Caption")
doc.add_heading("Maps of estimates and uncertainty",level=1).paragraph_format.page_break_before = True
add_figure(fig1,"Figure 1. Modelled 2024 deprivation and width of the credible interval.",
           "Two state maps showing the posterior mean and credible interval width in 2024.")
add_figure(fig2,"Figure 2. Modelled change and posterior probability of a decline.",
           "Two state maps showing 2018 to 2024 change and probability of a decline.")
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Bayesian_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved two verified 600 dpi JPEGs and all tables in {OUTPUT.resolve()}.")
